# **Objective**
In this notebook, I demonstrate that it is possible to find a compensating charge $z(\mathbf{r})$ for the electron charge $n(\mathbf{r})$, such that the Hartree potential of the sum of the two calculated with Ewald's method, $V_{H,\mathbf{G} \neq 0}[n+z](\mathbf{r})$, is 0 when $\mathbf{r} \notin U_a$, where $U_a$ is the region where $n$ and $z$ is localized.

To achieve the above, it suffices to make all the $l =2$ cartesian moments as well as all the spherical multipole moment of $n+z$ zero.

Note that making all $l=2$ cartesian moments zero necessarily makes all $l=2$ spherical moments zero. In fact,

$$
l=2\text{ cartesian moments zero}\\
⇔\\
l=2\text{ spherical moments zero} ∧ \int d\mathbf{r}\, r^2 [n+z](\mathbf{r}) = 0
$$

Therefore, we see that making $l=2$ cartesian moments zero induces some extra $l=0$ term, which then needs to be accounted for in the compensating charge $z$.

## **Constructing $n$**
Let's see start from the simplest case, where the electron charge is s-type
$$
n(\mathbf{r}) = N_1 e^{-\alpha_1 r^2}
$$
where $N_1$ is the normalization constant such that $q = \int d\mathbf{r}\, n(\mathbf{r})$. Explicitly,
$$
N_1 = q \left(\frac{\alpha_1}{\pi} \right) ^ {3/2}
$$

In [2]:
from pyscf.pbc import gto as pgto
import pyscf

import numpy as np

# cell params
L = 10
x = L/2
ke_cutoff = 200

# electron charge
alpha1 = 9.

basis = {'He': [[0, [alpha1, 1.]]]}

mol = pgto.M(
    atom = f'He {x} {x} {x}',
    basis = basis,
    ke_cutoff=ke_cutoff,
    a = np.eye(3)*L
)

# evaluate GTO vals on fft grid
mesh = pyscf.pbc.tools.cutoff_to_mesh(mol.lattice_vectors(), mol.ke_cutoff)
Rgrid = mol.get_uniform_grids(mesh=mesh, wrap_around=False)
aoOnR = mol.pbc_eval_gto('GTOval', Rgrid)
dv = mol.vol/Rgrid.shape[0]

# normalize properly
q = 1
N1 = q * (alpha1/np.pi)**(3/2) * (np.pi/alpha1/2)**(3/4)
nOnR = N1 * aoOnR[:, 0]

Test that $n$ as constructed is properly normalized

In [3]:
1 - np.sum(nOnR) * dv

np.float64(1.4553680482976006e-10)

### **Calculate the Ewald potential**

In [4]:
def get_Gv(nmesh,reciprocal_vecs):
    rx = np.fft.fftfreq(nmesh[0], 1./nmesh[0])
    ry = np.fft.fftfreq(nmesh[1], 1./nmesh[1])
    rz = np.fft.fftfreq(nmesh[2], 1./nmesh[2])
    return np.dot(pyscf.lib.cartesian_prod((rx,ry,rz)), reciprocal_vecs).astype(np.float64)

def getFormFactor(nmesh,cell):
    G2 = get_Gv(nmesh,cell.reciprocal_vectors())**2
    G2 = np.sum( G2, axis = 1)
    FF = np.zeros((G2.shape[0]), np.float64)
    idx = np.greater( G2, 0.)
    FF[idx] = 4. * np.pi /G2[idx]
    return FF

In [5]:
FF = getFormFactor(mesh, mol).reshape(mesh)
vnOnR = np.fft.ifftn(np.fft.fftn(nOnR.reshape(mesh)) * FF).real.flatten()

## **Constructing $z$**
We now construct the compensating charge $z$:
$$
z(\mathbf{r}) = N_2 e^{-\alpha_2 r^2}
$$
where $N_2$ is the normalization constant such that $\int d\mathbf{r} \, z(\mathbf{r}) = -q$

In [6]:
# spherical compensating charge
alpha2 = 4.

gbasis = {'He': [[0, [alpha2, 1.]]]}

gmol = pgto.M(
    atom = mol.atom,
    basis = gbasis,
    ke_cutoff=mol.ke_cutoff,
    a = np.eye(3)*L
)

# evaluate GTO vals on fft grid
gOnR = gmol.pbc_eval_gto('GTOval', Rgrid)

# normalize properly
q = 1
N2 = q * (alpha2/np.pi)**(3/2) * (np.pi/alpha2/2)**(3/4)
zOnR = N2 * gOnR[:, 0]

Check that $z$ is normalized

In [7]:
1 - zOnR.sum() * dv

np.float64(5.882216935759743e-11)

In [8]:
vzOnR = np.fft.ifftn(np.fft.fftn(zOnR.reshape(mesh)) * FF).real.flatten()

## **The Problem**
Now, as constructed, you might expect $V_{H, \mathbf{G} \neq 0} [n + z] (\mathbf{r}) = 0$ when $\mathbf{r} \notin U_a$, because all (spherical) multipole moment of $n+z$ vanishes (meaning that $V_{H, exact}(\mathbf{r}) = 0$ when $r \notin U_a$). However, this is NOT the case as shown below

In [9]:
# determine interstitial region
eps = 1.e-5
mask = zOnR < eps # mask for points outside of Ua

# the total hartree potential
vnzOnR = vnOnR - vzOnR

print(vnzOnR[mask].max())
print(vnzOnR[mask].min())

-6.408158225051919e-05
-6.466275442312819e-05


One can see from above that the potential far away is a non-zero constant. In fact, this constant is given by
$$
V_{H, \mathbf{G} \neq 0}[n+z] = V_{H, exact}[n+z] - V_{H, \mathbf{G} = 0}[n+z]
$$
where
$$
V_{H, \mathbf{G} = 0}[n+z](\mathbf{r}) = -\frac{4 \pi}{\Omega}\frac{1}{6}\int_{cell} d\mathbf{r} \, r^2 [n(\mathbf{r}) + z(\mathbf{r})]
$$
This does not have a simple analytical solution. In fact, the best I can do is to write:
$$
\int_{cell} d\mathbf{r} \, r^2 n(\mathbf{r}) = N_1 \sum_\mathbf{R} \int_{cell - \mathbf{R}} d\mathbf{r} \, (\mathbf{r} + \mathbf{R})^2 e^{-\alpha_1 (\mathbf{r} - \mathbf{r}_1)^2}
$$
So let's calculate this numerically:

In [10]:
def vG0(dens, Rgrid, dv, vol):
    assert(dens.shape[0] == Rgrid.shape[0])
    r2 = np.sum(Rgrid**2, axis=1)

    return -(4*np.pi/vol)*(1/6) * np.dot(dens, r2) * dv

Indeed, at the box edge (well outside $U_a$)
$$
V_{H, \mathbf{G} \neq 0}[n+z](\mathbf{r}) + V_{H, \mathbf{G} = 0}[n+z](\mathbf{r}) = 0 = V_{H, exact}[n+z](\mathbf{r})
$$

In [11]:
(vnOnR[0] - vzOnR[0]) + vG0(nOnR-zOnR, Rgrid, dv, mol.vol)

np.float64(1.087868636590484e-11)

## **The Solution**
As illustrated above, simply making all the (spherical) multipole moment zero is not enough to make $V_{H, \mathbf{G} = 0} (\mathbf{r}) = 0$ when $\mathbf{r}  \notin U_a$. What we need to do instead is the following:
1. Make all the cartesian $l=2$ moments of $[n+z](\mathbf{r})$ zero.
1. Make sure all spherical moments remain zero. This would result in a different $l=0$ component for $z$.

In this particular case, it suffices to consider $z(\mathbf{r})$ of the form:
$$
z(\mathbf{r}) = \left(C_0 + C_{2,x^2}x^2 + C_{2,y^2}y^2 + C_{2,z^2}z^2\right) e^{-\alpha_2 r^2}
$$
and solve the following set of equations:
$$
\begin{cases}
\int d\mathbf{r} \, x^2 [n+z](\mathbf{r}) = 0\\
\int d\mathbf{r} \, y^2 [n+z](\mathbf{r}) = 0\\
\int d\mathbf{r} \, z^2 [n+z](\mathbf{r}) = 0\\
\int d\mathbf{r} \, [n+z](\mathbf{r}) = 0
\end{cases}
$$
The first 3 equations allow us to write $C_{2, x^2} = C_{2, y^2} = C_{2, z^2} \equiv C_2$, that is
$$
z(\mathbf{r}) = \left(C_0 + C_{2}r^2 \right) e^{-\alpha_2 r^2}
$$
where $C_2$ can be written in terms of $C_0$. The last equation then allows us to solve for $C_0$ and $C_2$:
$$
\begin{cases}
C_0 = N_2 \left( \frac{3\alpha_2}{2\alpha_1} - \frac{5}{2}\right)\\
C_2 = N_2 \left( \alpha_2 - \frac{\alpha_2^2}{\alpha_1} \right)
\end{cases}
$$
Note that the above reduce to the trivial case when $\alpha_1 = \alpha_2$

In [ ]:
# cartesian compensating charge
alpha2 = 4
cbasis = {'He': [[0, [alpha2, 1.]], [2, [alpha2, 1.]]]}

cmol = pgto.M(
    atom = mol.atom,
    basis = cbasis,
    ke_cutoff=mol.ke_cutoff,
    a = np.eye(3)*L,
    cart = True,
)

smol = pgto.M(
    atom = mol.atom,
    basis = cbasis,
    ke_cutoff=mol.ke_cutoff,
    a = np.eye(3)*L,
    cart = False,
)

# evaluate GTO vals on fft grid
cOnR = cmol.pbc_eval_gto('GTOval', Rgrid)
sOnR = smol.pbc_eval_gto('GTOval', Rgrid)

from scipy.special import gamma
# normalize properly
def normalize_cart_gto(ao, l, alpha):
    # normalize primitive cartesian gto
    n = l + 1.5
    return ao * (gamma(n)/(2 * (2*alpha)**n))**(1/2)

cOnR_0 = cOnR[:, 0] * (np.pi/alpha2/2)**(3/4)
cOnR_0_test = normalize_cart_gto(cOnR[:, 0], 0, alpha2)
cOnR_xx = normalize_cart_gto(cOnR[:, 1], 2, alpha2)
cOnR_yy = normalize_cart_gto(cOnR[:, 4], 2, alpha2)
cOnR_zz = normalize_cart_gto(cOnR[:, 6], 2, alpha2)

NN2 = q * (alpha2/np.pi)**(3/2)

C0 = NN2 * (3*alpha2/2/alpha1 - 5/2)
C2 = NN2 * (alpha2 - alpha2**2/alpha1)
zOnR_cart = C0 * cOnR_0 + C2 * (cOnR_xx + cOnR_yy + cOnR_zz)

vcOnR = np.fft.ifftn(np.fft.fftn(zOnR_cart.reshape(mesh)) * FF).real.flatten()

In [26]:
print(sOnR[:, 0].sum())
print(cOnR[:, 0].sum())

386.9088297527389
386.9088297527389


In [22]:
print(cOnR_0.sum() * dv)
print(cOnR_0_test.sum() * dv)

0.6960409995630208
0.1963495408378124


In [23]:
(np.pi/4)**(3/2)

0.6960409996039635

In [12]:
# determine interstitial region
eps = 1.e-5
masks = cOnR < eps # mask for points outside of Ua
mask = np.logical_and.reduce(masks.T, axis=0)

vncOnR = vnOnR + vcOnR

print(vncOnR[mask].max())
print(vncOnR[mask].min())

4.274352150002159e-09
-9.774224490888272e-08


One can see that the potential $V_{H, \mathbf{G} \neq 0 } [n + z] (\mathbf{r}) = 0$ when $\mathbf{r} \notin U_a$, when $z(\mathbf{r})$ is constructed as above.

# **The General Case**
The general strategy is to implement this solution is the following:
1. For $l=0$ and $l=2$, use the cartesian GTO as compensating charge to fit the pair density $\phi_P \phi_Q$
1. For the rest of the angular momentum, use spherical GTO as compensating charge (as before).

First, let us consider the $l=0, x^2, y^2, z^2$ subspace of the cartesian compensating charge. This is the only non-trivial subspace that requires care because they are coupled with each other, as shown in the previous example. The compensating charge has the general form as follows:
$$
z(\mathbf{r}) = \sum_{g \in \{ S_{00}, x^2, y^2, z^2\}} M_{PQ, g} N_g g e^{\alpha r^2}
$$
where
$$
N_g = \left( \frac{\Gamma(l_g+1.5)}{2 (2\alpha_g)^{l_g+1.5}} \right) ^ {-\frac{1}{2}}
$$
which is just the basis_norm already implemented.

The system of equation that needs to be solved is:
$$
\begin{cases}
\frac{1}{6} N_P N_Q \bar{\Gamma}_{PQ}^{(2)} \left(2+ 3 \mathcal{N}_{22} \mathcal{C}^{22}_{L_PL_Q} - \mathcal{N}_{20} \mathcal{C}^{20}_{L_P L_Q}\right) = \sum_g M_{PQ, g} N_g I_{g \cdot x^2}\\
\frac{1}{6} N_P N_Q \bar{\Gamma}_{PQ}^{(2)} \left(2- 3 \mathcal{N}_{22} \mathcal{C}^{22}_{L_PL_Q} - \mathcal{N}_{20} \mathcal{C}^{20}_{L_P L_Q}\right) = \sum_g M_{PQ, g} N_g I_{g \cdot y^2}\\
\frac{1}{3} N_P N_Q \bar{\Gamma}_{PQ}^{(2)} \left(1+\mathcal{N}_{20} \mathcal{C}^{20}_{L_P L_Q}\right) = \sum_g M_{PQ, g} N_g I_{g \cdot z^2}\\
N_P N_Q \mathcal{N}_{00} \mathcal{C}^{00}_{L_P L_Q} \Gamma^{(0)}_{PQ} = \sum_g M_{PQ,g} N_g I_g
\end{cases}
$$
where
$$
\Gamma^{(n)}_{PQ} \equiv \int_0^\infty dr \, r^{l_{PQ}} e^{-\alpha_{PQ}r^2} = \frac{\Gamma \left((l_{PQ} + 1)/2\right)}{2\alpha_{PQ}^{(l_{PQ} + 1)/2}}\\
l_{PQ} \equiv n + 2 + l_P + l_Q, \alpha_{PQ} = \alpha_P + \alpha_Q
$$
and
$$
\mathcal{N_{00}} = \sqrt{4 \pi}\\
\mathcal{N_{20}} = 4\sqrt{\frac{\pi}{5}}\\
\mathcal{N_{22}} = 4\sqrt{\frac{\pi}{15}}\\
$$

In [ ]:
def normalize_cart_gto(ao, l):
    # normalize primitive cartesian gto
    n = l + 1.5
    return ao * (gamma(n)/(2 * (2*alpha2)**n))**(1/2)

In [33]:
from pyscf.paw import ClebschGordan
CG = ClebschGordan.RealCG
print(CG[0, 0, 0])
print(1/np.sqrt(4*np.pi))

0.28209479177387814
0.28209479177387814


# Scratch Work

In [14]:
cOnR_0.sum() * dv

np.float64(0.6960409995630206)

In [15]:
(np.pi/alpha2) ** (3/2)

0.6960409996039635

In [16]:
print(cmol.ao_labels())
print(smol.ao_labels())

['0 He 1s    ', '0 He 3dxx  ', '0 He 3dxy  ', '0 He 3dxz  ', '0 He 3dyy  ', '0 He 3dyz  ', '0 He 3dzz  ']
['0 He 1s    ', '0 He 3dxy  ', '0 He 3dyz  ', '0 He 3dz^2 ', '0 He 3dxz  ', '0 He 3dx2-y2']


In [17]:
s0 = cmol.intor('int1e_ovlp_sph')

In [18]:
c = cmol.cart2sph_coeff()
s1 = c.T.dot(cmol.intor('int1e_ovlp_cart')).dot(c)

In [19]:
print(abs(s1-s0).sum())

4.9926772078939e-16


In [20]:
c[:, 0]

array([1., 0., 0., 0., 0., 0., 0.])

In [21]:
0.5*np.sqrt(5/np.pi)

np.float64(0.6307831305050401)

In [22]:
zz = cOnR[:, -1] * ((225*np.pi/(2 * 128**2 * (alpha2)**7))**(1/4))
l=2
n=l+1.5
zzg = cOnR[:, -1] * (gamma(n)/(2 * (2*alpha2)**n))**(1/2)
print(zz.sum() * dv)
print(zzg.sum() * dv)

0.0870051249501553
0.0870051249501553


In [23]:
print(((225*np.pi/(2 * 128**2 * alpha2**5))**(1/2)))
print(gamma(n)/(2 * (2*alpha2)**n)*4)

0.004589773452083131
0.0045897734520831315


In [24]:
(0.5 * (np.pi**3/alpha2**5)**0.5)

0.08700512495049544

In [25]:
gamma(3.5)

np.float64(3.323350970447843)

In [26]:
15/8 * np.sqrt(np.pi)

np.float64(3.323350970447842)